## K-D Tree 
K-Dimensional Tree is a data structure which help us in efficiently finding the nearest neighbour to a point. If we have a set of points given in an array, and asked to find the nearest point to a target point :

<img src="./assets/k-d.png" width="700px">

A naive approach is to find the distance to all the points from the target and choose the smallest one. But doing so will result in time complexity of $O(N)$. If there are millions of datapoints, this appraoch would need million iterations.

### Splitting the graph
Simply considering the x axis we choose the middle point and divide it into two region left and right :

<img src="./assets/k-d_1.svg" width="700px">

Then from any one of the side we ignore x axis and only look at y-axis and choose the mid point and divide the region into upper and lower :

<img src="./assets/k-d_2.svg" width="700px">

We repeat the same process to get divided region

<img src="./assets/k-d_3.svg" width="700px">

### Splits as Binary Tree
We can represent this splits with the help of binary tree whose nodes holds the coordinate, each level of binary tree represent the alternating x-axis and y-axis splits

<img src="./assets/k-d_4.png" width="700px">

If we have 3 dimensions :

<img src="./assets/k-d_5.png" width="700px">

Hence we can do this for any k-dimensional data.

### Finding least distance point
<img src="./assets/k-d_6.svg" width="700px">

Few more edge case to test for :

<img src="./assets/k-d_7.svg" width="700px">

### Comparision in Binary Tree
<img src="./assets/k-d_8.svg" width="700px">

Now checking the edge cases :

<img src="./assets/k-d_9.svg" width="700px">

## Implementing the K-D Tree 

Given the following points, construct a 2-dimensional KD-tree and implement a function to search for the nearest point to a given query point.

Points:

$$ (3,6), (17,15), (13,15), (6,12), (9,1), (2,7), (10,19) $$

Query point:

$$ Q=(10,10) $$

Find the nearest point to \(Q\).

In [3]:
class KDNode:
    def __init__(self, point, left=None, right=None):
        self.point = point
        self.left = left
        self.right = right


class KDTree:
    def __init__(self, points):
        self.root = self.build(points, 0)

    def build(self, points, depth):
        if not points:
            return None

        axis = depth % 2
        points.sort(key=lambda p: p[axis])

        median = len(points) // 2

        return KDNode(
            points[median],
            self.build(points[:median], depth + 1),
            self.build(points[median + 1:], depth + 1)
        )

    def distance_squared(self, p1, p2):
        return sum((a - b) ** 2 for a, b in zip(p1, p2))

    def nearest(self, query):
        return self._nearest(self.root, query, 0, None, float("inf"))

    def _nearest(self, node, query, depth, best_point, best_distance):

        if node is None:
            return best_point, best_distance

        # Distance from query to current node
        distance = self.distance_squared(query, node.point)

        # Update best point
        if distance < best_distance:
            best_point = node.point
            best_distance = distance

        # Current splitting dimension
        axis = depth % 2

        # Decide which subtree to visit first
        if query[axis] < node.point[axis]:
            near = node.left
            far = node.right
        else:
            near = node.right
            far = node.left

        # Search the nearer subtree
        best_point, best_distance = self._nearest(
            near,
            query,
            depth + 1,
            best_point,
            best_distance
        )

        # Check whether the other subtree could contain
        # a closer point
        difference = query[axis] - node.point[axis]

        if difference ** 2 < best_distance:
            best_point, best_distance = self._nearest(
                far,
                query,
                depth + 1,
                best_point,
                best_distance
            )

        return best_point, best_distance

In [4]:
points = [
    (3, 6),
    (17, 15),
    (13, 15),
    (6, 12),
    (9, 1),
    (2, 7),
    (10, 19)
]

tree = KDTree(points)
query = (10, 10)

nearest_point, distance = tree.nearest(query)

print("Nearest point:", nearest_point)
print("Distance:", distance ** 0.5)

Nearest point: (6, 12)
Distance: 4.47213595499958


In [5]:
points = [
    (1, 2),
    (-8, 2),
    (2, -1),
    (-11, -6),
    (-9, 4),
    (6, -2),
    (7, 5),
    (-12,-3),
    (-4,-1),
    (-13,7),
    (-5,6),
    (5,-3),
    (8,-4),
    (3,0),
    (10,3)
]

tree = KDTree(points)
query = (-9, -3)

nearest_point, distance = tree.nearest(query)

print("Nearest point:", nearest_point)
print("Distance:", distance ** 0.5)

Nearest point: (-12, -3)
Distance: 3.0
